# SINaS & GLOBI backbone with GBIF matching

In this script, we match downloaded Sinas and GLOBI data to make sure they both contain the same taxonomic names/synonyms

## 1. Setup and loading of SINAS data

In [ ]:
import pandas as pd
from pathlib import Path
from pygbif import species as gbif_species
from tqdm import tqdm
import gzip
import re
import time
import requests
import asyncio
import aiohttp
from tqdm.asyncio import tqdm_asyncio
from pygbif import occurrences, species as gbif_species

In [2]:
# --- CONFIG ---
repo_root = Path.cwd().parent
file_path = repo_root / 'data' / 'SInAS_3.1.1.csv'
globi_path = repo_root / 'data' / 'interactions.tsv.gz'

# Load the raw data
raw_data = pd.read_csv(file_path, sep=None, engine='python', quotechar='"')
print(f"✅ Data loaded. Rows: {len(raw_data)}")

# Use the 'taxon' column for names
species_col = 'taxon'
# Clean names: strip whitespace and remove any trailing 'sp.' or author names if possible
unique_sinas_names = raw_data[species_col].str.strip().dropna().unique().tolist()
print(f"Unique species to harmonize: {len(unique_sinas_names)}")
unique_sinas_names[:10]

✅ Data loaded. Rows: 427956
Unique species to harmonize: 41171


['Aphaenogaster splendida',
 'Camponotus fallax',
 'Camponotus vagus',
 'Cardiocondyla mauritanica',
 'Cataglyphis nodus',
 'Crematogaster scutellaris',
 'Crematogaster sordidula',
 'Formica rufibarbis',
 'Hypoponera eduardi',
 'Hypoponera punctatissima']

## 2. Perform SINAS to GBIF backbone alignment

In [3]:
GBIF_URL = "https://api.gbif.org/v1/species/match"

async def match_one(session: aiohttp.ClientSession, name: str, semaphore: asyncio.Semaphore) -> tuple[str, dict | None]:
    async with semaphore:
        for attempt in range(3):  # retry up to 3x
            try:
                async with session.get(
                    GBIF_URL,
                    params={"name": name, "strict": "false"},
                    timeout=aiohttp.ClientTimeout(total=15),
                ) as resp:
                    if resp.status == 429:  # rate limited
                        await asyncio.sleep(2 ** attempt)
                        continue
                    resp.raise_for_status()
                    res = await resp.json()

                    if res.get("matchType") not in ("NONE", None):
                        return name, {
                            "canonical":  res.get("canonicalName"),
                            "gbif_id":    res.get("usageKey"),
                            "genus_id":    res.get("genusKey"),  
                            "family_id":   res.get("familyKey"),  
                            "match_type": res.get("matchType"),
                            "confidence": res.get("confidence"),
                        }
                    else:
                        return name, None

            except Exception as e:
                if attempt == 2:
                    print(f"\n  ⚠️  Failed '{name}' after 3 attempts: {e}")
                    return name, None
                await asyncio.sleep(1)

    return name, None


async def match_all(names: list[str], concurrency: int = 50) -> dict:
    semaphore = asyncio.Semaphore(concurrency)  # max concurrent requests
    connector = aiohttp.TCPConnector(limit=concurrency)

    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [match_one(session, name, semaphore) for name in names]
        results = await tqdm_asyncio.gather(*tasks, desc="Matching names to GBIF backbone")

    return dict(results)


# --- Run ---
# In a notebook, use await directly; in a plain .py script use asyncio.run()
sinas_map = await match_all(unique_sinas_names, concurrency=50)
# or if running as a script:
# sinas_map = asyncio.run(match_all(unique_sinas_names, concurrency=50))

# --- Map back ---
def get_field(name, field):
    entry = sinas_map.get(name)
    return entry[field] if entry else None

raw_data['gbif_canonical_name'] = raw_data['taxon'].map(lambda x: get_field(x, 'canonical'))
raw_data['gbif_id']             = raw_data['taxon'].map(lambda x: get_field(x, 'gbif_id'))
raw_data['gbif_genus_id']  = raw_data['taxon'].map(lambda x: get_field(x, 'genus_id'))
raw_data['gbif_family_id'] = raw_data['taxon'].map(lambda x: get_field(x, 'family_id'))
raw_data['gbif_match_type']     = raw_data['taxon'].map(lambda x: get_field(x, 'match_type'))
raw_data['gbif_confidence']     = raw_data['taxon'].map(lambda x: get_field(x, 'confidence'))

matched = raw_data['gbif_id'].notna().sum()
print(f"✅ Done. {matched}/{len(raw_data)} rows matched ({matched/len(raw_data):.1%})")
print(raw_data['gbif_match_type'].value_counts(dropna=False))

Matching names to GBIF backbone: 100%|██████████| 41171/41171 [00:34<00:00, 1203.99it/s]


✅ Done. 426805/427956 rows matched (99.7%)
gbif_match_type
EXACT         420632
HIGHERRANK      4806
FUZZY           1367
None            1151
Name: count, dtype: int64


## 3. Perform GLOBI to GBIF backbone alignment

In [4]:
#Harmonize GloBI names to GBIF backbone

globi_taxa = {}  # raw_globi_name → gbif_id

with gzip.open(globi_path, 'rt', encoding='utf-8') as f:
    header = f.readline().strip().split('\t')
    s_name_idx = header.index('sourceTaxonName')
    s_id_idx   = header.index('sourceTaxonId')
    t_name_idx = header.index('targetTaxonName')
    t_id_idx   = header.index('targetTaxonId')

    for line in tqdm(f, desc="Extracting GloBI taxa"):
        parts = line.strip().split('\t')
        for name_idx, id_idx in [(s_name_idx, s_id_idx), (t_name_idx, t_id_idx)]:
            if len(parts) <= id_idx:
                continue
            name   = parts[name_idx].strip()
            raw_id = parts[id_idx].strip()
            if not name or name in globi_taxa:
                continue
            m = re.search(r'GBIF:(\d+)', raw_id)
            globi_taxa[name] = int(m.group(1)) if m else None  # None = needs API lookup

already_resolved = sum(1 for v in globi_taxa.values() if v is not None)
needs_lookup     = [n for n, v in globi_taxa.items() if v is None]
print(f"GloBI taxa total:        {len(globi_taxa)}")
print(f"  Resolved from TSV:     {already_resolved}")
print(f"  Needs GBIF API lookup: {len(needs_lookup)}")

# --- Only call the API for names GloBI didn't already resolve ---
if needs_lookup:
    api_results = await match_all(needs_lookup, concurrency=50)
    for name, result in api_results.items():
        globi_taxa[name] = result['gbif_id'] if result else None

resolved = sum(1 for v in globi_taxa.values() if v is not None)
print(f"✅ GloBI harmonization done. {resolved}/{len(globi_taxa)} names resolved to GBIF IDs")

Extracting GloBI taxa: 20361182it [11:06, 30558.65it/s]


GloBI taxa total:        1040927
  Resolved from TSV:     122767
  Needs GBIF API lookup: 918160


Matching names to GBIF backbone: 100%|██████████| 918160/918160 [09:37<00:00, 1589.21it/s]


✅ GloBI harmonization done. 497445/1040927 names resolved to GBIF IDs


## 4. Verify overlap between Sinas & Globi resolution

In [7]:
# Create a set of the IDs you successfully got from GloBI
resolved_globi_ids = {v for v in globi_taxa.values() if v is not None}

# Check the overlap with your SInAS IDs (from your previous GBIF run)
sinas_ids = set(raw_data['gbif_id'].dropna().unique())
overlap = sinas_ids.intersection(resolved_globi_ids)

print(f"SInAS Species: {len(sinas_ids)}")
print(f"Interactions found for: {len(overlap)}")
print(f"Actual Useful Coverage: {(len(overlap)/len(sinas_ids))*100:.2f}%")

SInAS Species: 38586
Interactions found for: 26541
Actual Useful Coverage: 68.78%


## 5. Save the matched Sinas & Globi datasets

In [6]:
# ==============================================================================
# --- SAVE SUBSETS & EXTRACT NETWORK ---
# ==============================================================================

# 1. Save the filtered SInAS subset (only rows that have a GloBI match)
sinas_subset = raw_data[raw_data['gbif_id'].isin(overlap)].copy()
sinas_subset.to_csv(repo_root / 'data' / 'sinas_matched_species.csv', index=False)
print(f"📁 Saved {len(sinas_subset)} matched SInAS rows to 'sinas_matched_species.csv'")


# 2. Create and save the ID Bridge (Mapping SInAS names to GloBI names via GBIF ID)
# Convert the globi_taxa dict to a dataframe for easy merging
globi_mapping_df = pd.DataFrame([
    {'globi_taxon_name': k, 'gbif_id': v} 
    for k, v in globi_taxa.items() if v in overlap
])

# Merge with SInAS unique matched names to create a unified lookup table
sinas_names_df = raw_data[['taxon', 'gbif_id']].dropna().drop_duplicates()
id_bridge_df = pd.merge(sinas_names_df, globi_mapping_df, on='gbif_id', how='inner')
id_bridge_df.rename(columns={'taxon': 'sinas_taxon_name'}, inplace=True)

id_bridge_df.to_csv(repo_root / 'data' / 'sinas_globi_id_bridge.csv', index=False)
print(f"📁 Saved ID bridge table to 'sinas_globi_id_bridge.csv'")


# 3. Stream the GloBI TSV again to extract the actual interaction network
matched_network = []

with gzip.open(globi_path, 'rt', encoding='utf-8') as f:
    header = f.readline().strip().split('\t')
    
    # Identify the indices of columns we want to extract
    s_name_idx = header.index('sourceTaxonName')
    t_name_idx = header.index('targetTaxonName')
    interaction_idx = header.index('interactionTypeName')
    
    # Optional: grab study citations if you need them later
    path_idx = header.index('referenceCitation') if 'referenceCitation' in header else None

    for line in tqdm(f, desc="Extracting matched GloBI network edges"):
        parts = line.strip().split('\t')
        if len(parts) <= max(s_name_idx, t_name_idx):
            continue
            
        s_name = parts[s_name_idx].strip()
        t_name = parts[t_name_idx].strip()
        
        # Use your in-memory globi_taxa dict to find their GBIF IDs instantly
        s_gbif_id = globi_taxa.get(s_name)
        t_gbif_id = globi_taxa.get(t_name)
        
        # Keep the row if EITHER the source or target species belongs to your SInAS dataset
        if s_gbif_id in overlap or t_gbif_id in overlap:
            matched_network.append({
                'source_taxon_name': s_name,
                'source_gbif_id': s_gbif_id,
                'interaction_type': parts[interaction_idx] if interaction_idx < len(parts) else None,
                'target_taxon_name': t_name,
                'target_gbif_id': t_gbif_id,
                'reference': parts[path_idx] if path_idx and path_idx < len(parts) else None
            })

# Convert the network list to a DataFrame and save it
globi_network_df = pd.DataFrame(matched_network)
globi_network_df.to_csv(repo_root / 'data' / 'matched_globi_network.csv', index=False)

print(f"\n🚀 SUCCESS!")
print(f"Total network edges extracted: {len(globi_network_df)}")
print(f"Saved network data to 'matched_globi_network.csv'")

📁 Saved 378728 matched SInAS rows to 'sinas_matched_species.csv'
📁 Saved ID bridge table to 'sinas_globi_id_bridge.csv'


Extracting matched GloBI network edges: 20361182it [05:24, 62802.43it/s] 



🚀 SUCCESS!
Total network edges extracted: 9302662
Saved network data to 'matched_globi_network.csv'
